# Video2World · Quick Eval

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AetherLabsAI/Video2World/blob/main/notebooks/v2w_colab.ipynb)

Pick a mode in the first cell, then **Runtime → Run all**.

| Mode | What it does | Needs |
|---|---|---|
| `har` | Scores the human-assisted reference shipped with the data (checks that the evaluator and V2WScore work). ~1 min. | CPU runtime, no keys |
| `packages` | Evaluates one protocol package you already produced (upload a zip, Google Drive folder, or Hugging Face repo) and scores it. | GPU runtime recommended |
| `agent` | Runs a coding agent (Claude Code, Codex, or any OpenAI-compatible endpoint via OpenCode) on one instance, then evaluates and scores it. | GPU runtime, API key in Colab **Secrets** |
| `custom` | Runs your own agent command (`v2w run --agent-cmd`, see `docs/custom_agent.md`). | GPU runtime |

**Coverage.** Colab can run the ManiSkill/SAPIEN and MuJoCo tracks (FurnitureBench, DROID, hand, Push-T, twins, cloth: 160 of the 222 instances). The Isaac Sim tracks (RoboDojo and in-house, 62 instances) need a local GPU server with Vulkan; Colab evaluates **one instance** per run (a few minutes for scoring, up to the agent's own time for `agent`), so it is a try-out, not a leaderboard number. For the full V2WScore over all 222 instances, follow the README installation on your own machine.

**Keys.** Add the key your agent needs (`ANTHROPIC_API_KEY`, `OPENAI_API_KEY`, `OPENROUTER_API_KEY`, ...) under the 🔑 *Secrets* tab and enable notebook access. Keys are read from there and never written to the notebook.

In [ ]:
#@title 1 · Choose what to run
MODE = "har"  #@param ["har", "packages", "agent", "custom"]

#@markdown **Which instance.** Colab evaluates exactly one, so a run stays short.
TASK = "push-t"  #@param {type:"string"}
#@markdown A task family (`push-t`, `drawer-insert`, ...; first instance is used) or `family/sample`. Empty = first runnable instance. In mode `packages`, empty = the first package found.

#@markdown **Mode `packages`**
PACKAGES_SOURCE = "upload zip"  #@param ["upload zip", "google drive", "hugging face repo"]
PACKAGES_PATH = ""  #@param {type:"string"}
#@markdown Drive: folder path, e.g. `/content/drive/MyDrive/my_packages`. Hugging Face: dataset repo id. Layout: `<family>/<sample>/protocol.json`.

#@markdown **Mode `agent`**
AGENT = "claude"  #@param ["claude", "codex", "opencode (OpenAI-compatible endpoint)"]
MODEL = "claude-opus-5-5"  #@param {type:"string"}
BASE_URL = ""  #@param {type:"string"}
#@markdown Only for `opencode`: e.g. `https://openrouter.ai/api/v1`, or your vLLM/SGLang endpoint.
API_KEY_SECRET = "ANTHROPIC_API_KEY"  #@param {type:"string"}
#@markdown Name of the Colab secret holding the key (`OPENAI_API_KEY` for codex, `OPENROUTER_API_KEY` for OpenRouter).
REASONING_EFFORT = ""  #@param ["", "low", "medium", "high"]
WORKERS = 1  #@param {type:"slider", min:1, max:4, step:1}

#@markdown **Mode `custom`**
CUSTOM_CMD = "my_agent --brief {brief} --video {video} --out {out}"  #@param {type:"string"}
#@markdown Placeholders: `{task} {video} {brief} {out} {model} {profile} {family}`.

#@markdown **Run name** (results go to `runs/<name>/`)
RUN_NAME = "colab"  #@param {type:"string"}

In [ ]:
#@title 2 · Install
import os, subprocess, sys, shutil
def sh(cmd, check=True):
    r = get_ipython().system(cmd)
    return r

if not os.path.isdir("/content/Video2World"):
    sh("git clone --depth 1 https://github.com/AetherLabsAI/Video2World.git /content/Video2World")
os.chdir("/content/Video2World")
sh("pip install -q -e . huggingface_hub")

NEEDS_SIM = MODE != "har"
if NEEDS_SIM:
    # ManiSkill / SAPIEN tracks, with Vulkan for headless rendering
    sh("apt-get -qq install -y libvulkan1 mesa-vulkan-drivers libgl1 >/dev/null")
    sh("pip install -q -e .[sim]")
    sh("python -m mani_skill.utils.download_asset xarm6 -y")
    # MuJoCo tracks (twins, cloth) need their own interpreter
    if not os.path.isdir(".venv-twin"):
        sh("python -m venv .venv-twin && .venv-twin/bin/pip install -q -e .[twin]")
    sh("v2w setup --twin-python .venv-twin/bin/python")
if MODE == "agent":
    sh("npm install -g --silent @anthropic-ai/claude-code @openai/codex opencode-ai")
sh("v2w setup")

In [ ]:
#@title 3 · Download the data (only what the chosen mode needs)
from huggingface_hub import snapshot_download
from v2w import benchmark   # needs data/manifest.json: fetched first

from pathlib import Path
import zipfile
REPO = "AetherLabs-AI/Video2World"
def get_packages():
    PKG = "/content/packages"
    if PACKAGES_SOURCE == "upload zip":
        from google.colab import files
        name = next(iter(files.upload()))
        shutil.rmtree(PKG, ignore_errors=True); zipfile.ZipFile(name).extractall(PKG)
    elif PACKAGES_SOURCE == "google drive":
        from google.colab import drive; drive.mount("/content/drive"); PKG = PACKAGES_PATH
    else:
        snapshot_download(PACKAGES_PATH, repo_type="dataset", local_dir=PKG)
    return PKG
def fetch(patterns):
    snapshot_download(REPO, repo_type="dataset", local_dir="data", allow_patterns=patterns)

if MODE == "har":
    fetch(["manifest.json", "paths.json", "har/**", "LICENSE"])
else:
    fetch(["manifest.json", "paths.json", "har/**", "assets/**", "sources/**", "LICENSE"])
    COLAB_TRACKS = {"furniture", "hand", "twins", "cloth"}   # ManiSkill/SAPIEN and MuJoCo; robodojo and inhouse need Isaac Sim
    runnable = [c for c in benchmark.configurations() if c.track in COLAB_TRACKS]
    if MODE == "packages":
        PKG = get_packages()
        have = {(p.parent.parent.name, p.parent.name) for p in Path(PKG).glob("*/*/protocol.json")}
        runnable = [c for c in runnable if (c.family, c.sample) in have]
    fam, _, smp = TASK.strip().partition("/")
    chosen = [c for c in runnable if (not fam or c.family == fam) and (not smp or c.sample == smp)]
    if not chosen:
        raise SystemExit("No runnable instance for TASK=%r. Run `v2w list`; RoboDojo and in-house families need Isaac Sim." % TASK)
    ONE = chosen[0]
    fetch([f"samples/{ONE.sample}/**"])
    print("Evaluating:", ONE.key)

In [ ]:
#@title 4 · Run
import shlex, json
from google.colab import userdata

def selection():
    return f"--family {shlex.quote(ONE.family)} --sample {shlex.quote(ONE.sample)}"

def key_env(secret):
    try: os.environ[secret] = userdata.get(secret)
    except Exception:
        from getpass import getpass
        os.environ[secret] = getpass(f"{secret}: ")

import torch
GPUS = ",".join(str(i) for i in range(torch.cuda.device_count())) or "0"
base = f"--name {shlex.quote(RUN_NAME)} --gpus {GPUS} --workers {WORKERS} --video"

if MODE == "har":
    sh("v2w har --out runs/har_report")
    REPORT = "runs/har_report"

elif MODE == "packages":
    sh(f"v2w eval {base} --packages {shlex.quote(PKG)} {selection()}")
    REPORT = f"runs/{RUN_NAME}/report"

elif MODE == "agent":
    key_env(API_KEY_SECRET)
    kind = AGENT.split()[0]
    cmd = f"v2w run {base} --model {shlex.quote(MODEL)} {selection()}"
    if kind == "opencode":
        if not BASE_URL: raise SystemExit("opencode needs BASE_URL (OpenAI-compatible endpoint).")
        cmd += f" --base-url {shlex.quote(BASE_URL)} --api-key-env {API_KEY_SECRET}"
    else:
        cmd += f" --agent {kind}"
        if kind == "codex" and BASE_URL: cmd += f" --base-url {shlex.quote(BASE_URL)} --api-key-env {API_KEY_SECRET}"
    if REASONING_EFFORT: cmd += f" --reasoning-effort {REASONING_EFFORT}"
    sh(cmd)
    REPORT = f"runs/{RUN_NAME}/report"

else:
    sh(f"v2w run {base} --model {shlex.quote(MODEL)} --agent-cmd {shlex.quote(CUSTOM_CMD)} {selection()}")
    REPORT = f"runs/{RUN_NAME}/report"

In [ ]:
#@title 5 · Results
import base64, glob, html
import matplotlib.pyplot as plt
from IPython.display import HTML, display

INK, MUTED = "#2b2118", "#8a7d70"
DIM = {"F": ("#4f6f7d", "Function"), "G": ("#7c9a6d", "Geometry"), "D": ("#b0794f", "Dynamics")}
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.edgecolor": MUTED, "text.color": INK,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": INK, "font.size": 10})

summary = json.load(open(f"{REPORT}/summary.json"))
rows = json.load(open(f"{REPORT}/samples.json"))

def card(title, subtitle, score, parts):
    chips = "".join(f'<div style="flex:1"><div style="font:12px sans-serif;color:{MUTED}">{DIM[k][1]}</div>'
                    f'<div style="font:600 24px serif;color:{DIM[k][0]}">{v:.2f}</div></div>' for k, v in parts.items())
    display(HTML(f'<div style="border:1px solid #d9cfc3;border-radius:12px;padding:18px 22px;max-width:640px;background:#faf7f2;color:{INK}">'
                 f'<div style="font:12px sans-serif;color:{MUTED};letter-spacing:.08em;text-transform:uppercase">{html.escape(subtitle)}</div>'
                 f'<div style="font:600 18px sans-serif;margin:2px 0 10px">{html.escape(title)}</div>'
                 f'<div style="display:flex;gap:24px;align-items:flex-end"><div><div style="font:12px sans-serif;color:{MUTED}">Score</div>'
                 f'<div style="font:600 44px serif;line-height:1">{score:.1f}</div></div>{chips}</div></div>'))

def video_html(path, caption):
    if not path or not os.path.exists(path) or os.path.getsize(path) > 25e6:
        return ""
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    return (f'<figure style="margin:0"><video src="data:video/mp4;base64,{b64}" width="360" controls loop muted></video>'
            f'<figcaption style="font:12px sans-serif;color:{MUTED}">{caption}</figcaption></figure>')

if MODE == "har":
    card("Human-assisted reference", f"{summary['configurations']} instances · {summary['families']} families",
         summary["v2w_score"], {k: summary["dimensions"][k] for k in "FGD"} if all(k in summary["dimensions"] for k in "FGD") else {})
    fam = sorted(summary["by_family"].items(), key=lambda kv: kv[1]["score"])
    names = [k for k, _ in fam]; vals = [v["score"] for _, v in fam]
    fig, ax = plt.subplots(figsize=(7, 0.22 * len(names) + 1))
    ax.barh(names, vals, color="#5b4636", height=0.65); ax.set_xlim(0, 100); ax.set_xlabel("V2WScore")
    for sp in ("top", "right"): ax.spines[sp].set_visible(False)
    ax.set_title("Reference score per task family", loc="left", fontsize=11); plt.tight_layout(); plt.show()
else:
    r = next(x for x in rows if x["family"] == ONE.family and x["sample"] == ONE.sample)
    card(ONE.key, "one instance · try-out, not a leaderboard score", 100 * (r.get("S") or 0),
         {k: (r.get(k) or 0) for k in "FGD"})
    if not r.get("build"):
        print("The package did not build (build = false), so the score is 0. See the evaluator log under", f"runs/{RUN_NAME}/{ONE.family}/{ONE.sample}/")
    else:
        m = [(g, n, d) for g, grp in (("G", r.get("geometry") or {}), ("D", r.get("dynamics") or {})) for n, d in grp.items()]
        fig, ax = plt.subplots(figsize=(7, 0.45 * len(m) + 1.2))
        for i, (g, n, d) in enumerate(m):
            ax.barh(i, 1, color="#efe9e0", height=0.6)
            ax.barh(i, d["quality"], color=DIM[g][0], height=0.6)
            ax.text(1.02, i, f"{d['error']:.2f} / τ {d['tau']:g}", va="center", fontsize=9, color=MUTED)
        ax.set_yticks(range(len(m))); ax.set_yticklabels([n.replace("own_trajectory.", "") for _, n, _ in m]); ax.invert_yaxis()
        ax.set_xlim(0, 1); ax.set_xlabel("quality = max(0, 1 − error / τ)   (1 is perfect)")
        for sp in ("top", "right"): ax.spines[sp].set_visible(False)
        ax.set_title("Metric quality (green: geometry, orange: dynamics)", loc="left", fontsize=11); plt.tight_layout(); plt.show()
    src_video = str(Path("data/samples") / ONE.sample / "video.mp4")
    sim_video = next(iter(sorted(glob.glob(f"runs/{RUN_NAME}/{ONE.family}/{ONE.sample}/eval/video/*.mp4"))), None)
    display(HTML('<div style="display:flex;gap:16px;flex-wrap:wrap">' + video_html(src_video, "source video (the agent's only input)")
                 + video_html(sim_video, "your world, executed by the evaluator") + "</div>"))

shutil.make_archive("/content/v2w_report", "zip", REPORT)
try:
    from google.colab import files; files.download("/content/v2w_report.zip")
except Exception: pass